In [74]:
import os
import json
import pandas as pd

from collections import defaultdict, Counter
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

In [6]:
with open('data/raw/instances_default.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
data

{'licenses': [{'name': '', 'id': 0, 'url': ''}],
 'info': {'contributor': '',
  'date_created': '',
  'description': '',
  'url': '',
  'version': '',
  'year': ''},
 'categories': [{'id': 1, 'name': 'Seller', 'supercategory': ''},
  {'id': 2, 'name': 'Invoice_number', 'supercategory': ''},
  {'id': 3, 'name': 'Invoice_date', 'supercategory': ''},
  {'id': 4, 'name': 'Total_amount', 'supercategory': ''},
  {'id': 5, 'name': 'Item_description', 'supercategory': ''}],
 'images': [{'id': 1,
   'width': 1654,
   'height': 2339,
   'file_name': 'invoice_001.jpg',
   'license': 0,
   'flickr_url': '',
   'coco_url': '',
   'date_captured': 0},
  {'id': 2,
   'width': 1654,
   'height': 2339,
   'file_name': 'invoice_002.jpg',
   'license': 0,
   'flickr_url': '',
   'coco_url': '',
   'date_captured': 0},
  {'id': 3,
   'width': 1654,
   'height': 2339,
   'file_name': 'invoice_003.jpg',
   'license': 0,
   'flickr_url': '',
   'coco_url': '',
   'date_captured': 0},
  {'id': 4,
   'width': 

### Маппинг

In [8]:
annotations = data['annotations']
images = data['images']
categories = data['categories']

In [9]:
annotations

[{'id': 1,
  'image_id': 1,
  'category_id': 1,
  'segmentation': [],
  'area': 156929.9087,
  'bbox': [127.67, 437.45, 532.67, 294.61],
  'iscrowd': 0,
  'attributes': {'occluded': False, 'rotation': 0.0}},
 {'id': 2,
  'image_id': 1,
  'category_id': 5,
  'segmentation': [],
  'area': 371547.91339999996,
  'bbox': [135.2, 824.3, 1387.46, 267.79],
  'iscrowd': 0,
  'attributes': {'occluded': False, 'rotation': 0.0}},
 {'id': 3,
  'image_id': 1,
  'category_id': 4,
  'segmentation': [],
  'area': 66364.75040000002,
  'bbox': [484.57, 1320.82, 1029.23, 64.48],
  'iscrowd': 0,
  'attributes': {'occluded': False, 'rotation': 0.0}},
 {'id': 4,
  'image_id': 1,
  'category_id': 3,
  'segmentation': [],
  'area': 63760.205,
  'bbox': [115.76, 124.99, 892.75, 71.42],
  'iscrowd': 0,
  'attributes': {'occluded': False, 'rotation': 0.0}},
 {'id': 5,
  'image_id': 1,
  'category_id': 2,
  'segmentation': [],
  'area': 28643.720999999998,
  'bbox': [120.7, 50.2, 445.47, 64.3],
  'iscrowd': 0,
  '

In [10]:
images

[{'id': 1,
  'width': 1654,
  'height': 2339,
  'file_name': 'invoice_001.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 2,
  'width': 1654,
  'height': 2339,
  'file_name': 'invoice_002.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 3,
  'width': 1654,
  'height': 2339,
  'file_name': 'invoice_003.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 4,
  'width': 1654,
  'height': 2339,
  'file_name': 'invoice_004.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 5,
  'width': 1654,
  'height': 2339,
  'file_name': 'invoice_005.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 6,
  'width': 1654,
  'height': 2339,
  'file_name': 'invoice_006.jpg',
  'license': 0,
  'flickr_url': '',
  'coco_url': '',
  'date_captured': 0},
 {'id': 7,
  'width': 1654,
  'height': 2339,
  'file_name

In [12]:
categories

[{'id': 1, 'name': 'Seller', 'supercategory': ''},
 {'id': 2, 'name': 'Invoice_number', 'supercategory': ''},
 {'id': 3, 'name': 'Invoice_date', 'supercategory': ''},
 {'id': 4, 'name': 'Total_amount', 'supercategory': ''},
 {'id': 5, 'name': 'Item_description', 'supercategory': ''}]

In [14]:
image_id_to_name = {
    img['id']: img['file_name'] for img in images
}

image_id_to_name

{1: 'invoice_001.jpg',
 2: 'invoice_002.jpg',
 3: 'invoice_003.jpg',
 4: 'invoice_004.jpg',
 5: 'invoice_005.jpg',
 6: 'invoice_006.jpg',
 7: 'invoice_007.jpg',
 8: 'invoice_008.jpg',
 9: 'invoice_009.jpg',
 10: 'invoice_010.jpg',
 11: 'invoice_011.jpg',
 12: 'invoice_012.jpg',
 13: 'invoice_013.jpg',
 14: 'invoice_014.jpg',
 15: 'invoice_015.jpg',
 16: 'invoice_016.jpg',
 17: 'invoice_017.jpg',
 18: 'invoice_018.jpg',
 19: 'invoice_019.jpg',
 20: 'invoice_020.jpg',
 21: 'invoice_021.jpg',
 22: 'invoice_022.jpg',
 23: 'invoice_023.jpg',
 24: 'invoice_024.jpg',
 25: 'invoice_025.jpg',
 26: 'invoice_026.jpg',
 27: 'invoice_027.jpg',
 28: 'invoice_028.jpg',
 29: 'invoice_029.jpg',
 30: 'invoice_030.jpg',
 31: 'invoice_031.jpg',
 32: 'invoice_032.jpg',
 33: 'invoice_033.jpg',
 34: 'invoice_034.jpg',
 35: 'invoice_035.jpg',
 36: 'invoice_036.jpg',
 37: 'invoice_037.jpg',
 38: 'invoice_038.jpg',
 39: 'invoice_039.jpg',
 40: 'invoice_040.jpg',
 41: 'invoice_041.jpg',
 42: 'invoice_042.jpg',
 

In [15]:
category_id_to_name = {
    category['id']: category['name'] for category in categories
}

category_id_to_name

{1: 'Seller',
 2: 'Invoice_number',
 3: 'Invoice_date',
 4: 'Total_amount',
 5: 'Item_description'}

In [67]:
label2id = {
    category['name']: category['id'] - 1 for category in categories
}

label2id

{'Seller': 0,
 'Invoice_number': 1,
 'Invoice_date': 2,
 'Total_amount': 3,
 'Item_description': 4}

In [25]:
ALLOWED_LABELS = {
    category['name'] for category in categories
}

ALLOWED_LABELS

{'Invoice_date',
 'Invoice_number',
 'Item_description',
 'Seller',
 'Total_amount'}

### Группировка аннотаций по картинкам

In [17]:
dataset = defaultdict(lambda: {
    'bboxes': [],
    'labels': []
})

for ann in annotations:
    image_id = ann['image_id']
    bbox = ann['bbox']
    label = category_id_to_name[ann['category_id']]

    dataset[image_id]['bboxes'].append(bbox)
    dataset[image_id]['labels'].append(label)

dataset

defaultdict(<function __main__.<lambda>()>,
            {1: {'bboxes': [[127.67, 437.45, 532.67, 294.61],
               [135.2, 824.3, 1387.46, 267.79],
               [484.57, 1320.82, 1029.23, 64.48],
               [115.76, 124.99, 892.75, 71.42],
               [120.7, 50.2, 445.47, 64.3]],
              'labels': ['Seller',
               'Item_description',
               'Total_amount',
               'Invoice_date',
               'Invoice_number']},
             2: {'bboxes': [[111.0, 42.84, 457.81, 77.16],
               [477.9, 1956.12, 1035.87, 55.58],
               [127.4, 824.0, 1397.48, 898.72],
               [126.43, 431.21, 550.4, 300.92],
               [103.28, 122.57, 907.9, 74.59]],
              'labels': ['Invoice_number',
               'Total_amount',
               'Item_description',
               'Seller',
               'Invoice_date']},
             3: {'bboxes': [[125.29, 63.1, 449.35, 53.56],
               [128.27, 119.64, 883.82, 71.42],
          

In [20]:
final_dataset = []

for image_id, data_item in dataset.items():
    final_dataset.append({
        'image_path': f'images/{image_id_to_name[image_id]}',
        'bboxes': data_item['bboxes'],
        'labels': data_item['labels']
    })
    
final_dataset[0]

{'image_path': 'images/invoice_001.jpg',
 'bboxes': [[127.67, 437.45, 532.67, 294.61],
  [135.2, 824.3, 1387.46, 267.79],
  [484.57, 1320.82, 1029.23, 64.48],
  [115.76, 124.99, 892.75, 71.42],
  [120.7, 50.2, 445.47, 64.3]],
 'labels': ['Seller',
  'Item_description',
  'Total_amount',
  'Invoice_date',
  'Invoice_number']}

### Проверка консистентности датасета

In [60]:
def check_dataset_consistency(final_dataset, base_dir="data/raw"):
    problems = {
        'missing_keys': [],
        'length_mismatch': [],
        'file_not_found': [],
        'image_open_error': [],
        'empty_annotations': [],
        'invalid_bbox_format': [],
        'bbox_out_of_bounds': [],
        'invalid_labels':[],
    }

    class_counter = Counter()
    images_per_class = Counter()

    for idx, sample in enumerate(final_dataset):
        sample_id = f'sample_{idx}'

        # Проверка ключей
        required_keys = {'image_path', 'bboxes', 'labels'}
        missing = required_keys - set(sample.keys())
        if missing:
            problems['missing_keys'].append((sample_id, missing))
            continue

        image_path = Path(base_dir) / sample['image_path']
        bboxes = sample['bboxes']
        labels = sample['labels']

        # Проверка длины массива bboxes и длины массива labels
        if len(bboxes) != len(labels):
            problems['length_mismatch'].append((str(image_path), len(bboxes), len(labels)))

        # Проверка файлов на ошибку
        if not image_path.exists():
            problems['file_not_found'].append(str(image_path))

        # Проверка файла на битость
        try:
            with Image.open(image_path) as img:
                width, height = img.size
        except Exception as e:
            problems['image_open_error'].append((str(image_path), str(e)))
            continue

        # Проверка пустых аннотаций
        if len(bboxes) == 0 or len(labels) == 0:
            problems['empty_annotations'].append(str(image_path))

        # Проверка bbox
        sample_classes = set()

        for bbox, label in zip(bboxes, labels):
            # labels
            if label not in ALLOWED_LABELS:
                problems['invalid_labels'].append((str(image_path), label))

            class_counter[label] += 1
            sample_classes.add(label)

            # bbox format
            if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
                problems['invalid_bbox_format'].append((str(image_path), bbox))
                continue

            x, y, w, h = bbox

            # bbox bounds
            if x + w > width or y + h > height:
                problems["bbox_out_of_bounds"].append(
                    (str(image_path), bbox, (width, height))
                )
                
        for cls in sample_classes:
            images_per_class[cls] += 1
            
    return problems, class_counter, images_per_class

In [61]:
problems, class_counter, images_per_class = check_dataset_consistency(final_dataset)

print("=== Problems ===")
for issue_name, items in problems.items():
    print(f"{issue_name}: {len(items)}")

print("\n=== OBJECT COUNT BY CLASS ===")
for cls, count in class_counter.items():
    print(f"{cls}: {count}")

print("\n=== IMAGES COUNT BY CLASS ===")
for cls, count in images_per_class.items():
    print(f"{cls}: {count}")

=== Problems ===
missing_keys: 0
length_mismatch: 0
file_not_found: 0
image_open_error: 0
empty_annotations: 0
invalid_bbox_format: 0
bbox_out_of_bounds: 0
invalid_labels: 0

=== OBJECT COUNT BY CLASS ===
Seller: 1500
Item_description: 1500
Total_amount: 1498
Invoice_date: 1484
Invoice_number: 1484

=== IMAGES COUNT BY CLASS ===
Invoice_number: 1484
Seller: 1500
Item_description: 1500
Invoice_date: 1484
Total_amount: 1498


In [63]:
def find_duplicate_bboxes(final_dataset):
    duplicates = []

    for sample in final_dataset:
        image_path = sample['image_path']
        seen = set()

        for bbox, label in zip(sample['bboxes'], sample['labels']):
            key = (round(bbox[0], 2), round(bbox[1], 2), round(bbox[2], 2), round(bbox[3], 2), label)
            if key in seen:
                duplicates.append((image_path, bbox, label))
            seen.add(key)

    return duplicates

In [64]:
duplicates = find_duplicate_bboxes(final_dataset)
duplicates[:10]

[]

- пути к файлам валидны
- картинки открываются
- lengths совпадают
- bbox не выходят за границы
- labels корректные
- пустых аннотаций нет
- дублей нет

### Кодирование классов

In [72]:
final_dataset_encoded = []

for sample in final_dataset:
    encoded_sample = {
        "image_path": sample["image_path"],
        "bboxes": sample["bboxes"],
        "labels": [label2id[label] for label in sample["labels"]]
    }
    final_dataset_encoded.append(encoded_sample)

final_dataset_encoded[0]

{'image_path': 'images/invoice_001.jpg',
 'bboxes': [[127.67, 437.45, 532.67, 294.61],
  [135.2, 824.3, 1387.46, 267.79],
  [484.57, 1320.82, 1029.23, 64.48],
  [115.76, 124.99, 892.75, 71.42],
  [120.7, 50.2, 445.47, 64.3]],
 'labels': [0, 4, 3, 2, 1]}

### Разбивает на train, test, val

In [77]:
ids = list(range(len(final_dataset_encoded)))

train_idx, temp_idx = train_test_split(
    ids,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

train_data = [final_dataset_encoded[i] for i in train_idx]
val_data = [final_dataset_encoded[i] for i in val_idx]
test_data = [final_dataset_encoded[i] for i in test_idx]

print(len(train_data), len(val_data), len(test_data))

1050 225 225


In [78]:
split_dir = Path('data/interim')
split_dir.mkdir(parents=True, exist_ok=True)

with open(split_dir / 'train_idx.json', 'w', encoding='utf-8') as f:
    json.dump(train_idx, f)

with open(split_dir / 'val_idx.json', 'w', encoding='utf-8') as f:
    json.dump(val_idx, f)

with open(split_dir / "test_idx.json", "w", encoding="utf-8") as f:
    json.dump(test_idx, f)